# 04 — Modelling & Business Results

> **Insiders Loyalty Program** · Customer Segmentation Project

---

Géron (Ch. 8): *"Clustering algorithms try to detect groups of similar instances. K-Means is a simple algorithm that can cluster data quickly and efficiently, often in just a few iterations."*

This notebook covers:
1. K selection — Elbow method and Silhouette analysis
2. Train best K-Means model via the production pipeline
3. Cluster profiling (RFM stats per cluster)
4. PCA visualisation of clusters
5. Business translation — identify the Insiders cluster
6. Revenue analysis per cluster

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from insiders_loyalty_program.config import load_config
from insiders_loyalty_program.data import load_training_frame
from insiders_loyalty_program.features import build_rfm_features

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

config  = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
SEED    = int(config.get('modeling', {}).get('random_state', 42))
K_RANGE = config.get('modeling', {}).get('cluster_range', list(range(3, 10)))
print(f'K range: {K_RANGE}  |  Random state: {SEED}')

## 1. Prepare Feature Matrix

In [ ]:
df_raw = load_training_frame(config)
rfm    = build_rfm_features(df_raw, config)

rfm_features = [c for c in ['recency_days', 'frequency', 'monetary', 'avg_ticket', 'total_items']
                if c in rfm.columns]

preproc = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('log',    FunctionTransformer(np.log1p, validate=True)),
    ('scale',  StandardScaler()),
])

X_raw  = rfm[rfm_features].values
X      = preproc.fit_transform(X_raw)

print(f'Feature matrix shape: {X.shape}')

## 2. K Selection — Elbow Curve & Silhouette Score

We evaluate several values of `k` using three metrics:
- **Inertia** (within-cluster sum of squares) — decreases as k grows; look for the "elbow"
- **Silhouette Score** — peaks at the best k (higher = better separation)
- **Davies-Bouldin Index** — bottoms at the best k (lower = better)

In [ ]:
results = []
SAMPLE  = min(10_000, len(X))
rng     = np.random.default_rng(SEED)
idx_sil = rng.choice(len(X), size=SAMPLE, replace=False)

for k in K_RANGE:
    km     = KMeans(n_clusters=k, n_init='auto', random_state=SEED)
    labels = km.fit_predict(X)
    sil    = silhouette_score(X[idx_sil], labels[idx_sil])
    db     = davies_bouldin_score(X, labels)
    ch     = calinski_harabasz_score(X, labels)
    results.append({'k': k, 'inertia': km.inertia_,
                    'silhouette': sil, 'davies_bouldin': db, 'calinski_harabasz': ch})
    print(f'  k={k}  inertia={km.inertia_:>10.0f}  silhouette={sil:.3f}  db={db:.3f}')

results_df = pd.DataFrame(results)
print('\nDone.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(results_df['k'], results_df['inertia'], marker='o', linewidth=2, color='steelblue')
axes[0].set_title('Elbow Curve (Inertia)')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_xticks(K_RANGE)

best_k_sil = results_df.loc[results_df['silhouette'].idxmax(), 'k']
axes[1].plot(results_df['k'], results_df['silhouette'], marker='o', linewidth=2, color='darkorange')
axes[1].axvline(best_k_sil, color='red', linestyle='--', linewidth=1.5, label=f'Best k={best_k_sil}')
axes[1].set_title('Silhouette Score (higher = better)')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(K_RANGE)
axes[1].legend()

axes[2].plot(results_df['k'], results_df['davies_bouldin'], marker='o', linewidth=2, color='green')
axes[2].axvline(best_k_sil, color='red', linestyle='--', linewidth=1.5, label=f'Best k={best_k_sil}')
axes[2].set_title('Davies-Bouldin Index (lower = better)')
axes[2].set_xlabel('Number of Clusters (k)')
axes[2].set_ylabel('Davies-Bouldin')
axes[2].set_xticks(K_RANGE)
axes[2].legend()

plt.suptitle('K-Means Cluster Quality Metrics', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / '04_cluster_selection.png', dpi=120)
plt.show()
print(f'Best k by silhouette: {best_k_sil}')

## 3. Train Best Model via Production Pipeline

We now run the full training pipeline (same code used in `make train`) to produce the serialised model and the cluster assignment file.

In [ ]:
RUN_PIPELINE = True  # set False to skip re-training if already done

if RUN_PIPELINE:
    from insiders_loyalty_program.models import train_clustering
    result = train_clustering(config)
    print('Training complete.')
    print(f'  Model saved to : {result["model_path"]}')
    print(f'  Metrics saved  : {result["metrics_path"]}')
    print(f'  Best k         : {result["metrics"]["k"]}')
    print(f'  Silhouette     : {result["metrics"]["silhouette"]:.3f}')
    print(f'  Davies-Bouldin : {result["metrics"]["davies_bouldin"]:.3f}')

## 4. Load Results and Profile Clusters

In [ ]:
clusters_path = PROJECT_ROOT / 'reports' / 'cluster_assignments.csv'
metrics_path  = PROJECT_ROOT / 'reports' / 'metrics.json'

if not clusters_path.exists():
    print('Run training first (set RUN_PIPELINE=True above).')
else:
    df_clusters = pd.read_csv(clusters_path)
    metrics     = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}

    print(f'Cluster assignments shape: {df_clusters.shape}')
    print(f'\nCluster distribution:')
    print(df_clusters['cluster'].value_counts().sort_index().to_string())
    print(f'\nSelected metrics: {metrics.get("selected", {})}')

In [ ]:
if clusters_path.exists():
    agg_cols = [c for c in ['recency_days', 'frequency', 'monetary', 'avg_ticket', 'total_items']
                if c in df_clusters.columns]

    profile = df_clusters.groupby('cluster')[agg_cols].agg(['mean', 'median', 'count']).round(2)
    profile.columns = ['_'.join(c) for c in profile.columns]

    # Customer count per cluster
    profile['n_customers'] = df_clusters.groupby('cluster').size()
    profile['pct_customers'] = (profile['n_customers'] / len(df_clusters) * 100).round(1)

    if 'monetary_mean' in profile.columns:
        profile['revenue_share_%'] = (
            df_clusters.groupby('cluster')['monetary'].sum() /
            df_clusters['monetary'].sum() * 100
        ).round(1)

    print('Cluster Profile Summary:')
    display(profile[['n_customers', 'pct_customers'] +
                     [c for c in profile.columns if '_median' in c or c == 'revenue_share_%']])

## 5. Cluster Visualisation with PCA

We reduce the feature space to 2 dimensions using **Principal Component Analysis** (Ch. 7 of the book) to visualise the clusters. PCA finds the directions of maximum variance in the data.

In [ ]:
if clusters_path.exists() and len(agg_cols) >= 2:
    X_for_pca  = preproc.fit_transform(rfm[rfm_features].values)
    pca        = PCA(n_components=2, random_state=SEED)
    X_2d       = pca.fit_transform(X_for_pca)
    ev_ratio   = pca.explained_variance_ratio_

    labels = df_clusters['cluster'].values if len(df_clusters) == len(rfm) else None

    if labels is not None:
        n_clusters = len(np.unique(labels))
        colors     = plt.cm.tab10(np.linspace(0, 1, n_clusters))

        fig, ax = plt.subplots(figsize=(10, 7))
        for i, k in enumerate(np.unique(labels)):
            mask = labels == k
            ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                       c=[colors[i]], label=f'Cluster {k}',
                       s=20, alpha=0.55, edgecolors='none')

        ax.set_xlabel(f'PC1 ({ev_ratio[0]*100:.1f}% variance)')
        ax.set_ylabel(f'PC2 ({ev_ratio[1]*100:.1f}% variance)')
        ax.set_title(f'K-Means Clusters in PCA Space (k={n_clusters})')
        ax.legend(title='Cluster', markerscale=2)
        plt.tight_layout()
        plt.savefig(FIGURES / '04_clusters_pca.png', dpi=120)
        plt.show()
        print(f'PCA explains {ev_ratio.sum()*100:.1f}% of total variance in 2 components.')

## 6. Business Translation — Identifying the Insiders

The Insiders cluster is defined as the group with:
- **Lowest recency** (bought most recently)
- **Highest frequency** (most orders)
- **Highest monetary** (most revenue generated)

In [ ]:
if clusters_path.exists() and 'monetary' in df_clusters.columns:
    cluster_means = df_clusters.groupby('cluster')[agg_cols].mean()

    # Rank clusters: score = high monetary + high frequency + low recency
    cluster_means['score'] = (
        cluster_means.get('monetary', 0) +
        cluster_means.get('frequency', 0) * 100 -
        cluster_means.get('recency_days', 0)
    )
    insiders_cluster = cluster_means['score'].idxmax()

    labels_map = {}
    ranked = cluster_means.sort_values('score', ascending=False)
    for rank, cluster_id in enumerate(ranked.index):
        if rank == 0:
            labels_map[cluster_id] = 'INSIDERS'
        elif rank == len(ranked) - 1:
            labels_map[cluster_id] = 'Churned'
        elif rank == 1:
            labels_map[cluster_id] = 'Loyalists'
        else:
            labels_map[cluster_id] = 'At-Risk'

    df_clusters['segment'] = df_clusters['cluster'].map(labels_map)
    print(f'Cluster {insiders_cluster} → INSIDERS')
    print()
    print('Segment assignments:')
    print(df_clusters['segment'].value_counts().to_string())

In [ ]:
if clusters_path.exists() and 'monetary' in df_clusters.columns:
    rev_by_seg = df_clusters.groupby('segment')['monetary'].sum().sort_values(ascending=False)
    total_rev  = rev_by_seg.sum()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Revenue share by segment
    pct = (rev_by_seg / total_rev * 100).round(1)
    colors = ['gold' if s == 'INSIDERS' else 'steelblue' for s in pct.index]
    pct.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
    axes[0].set_title('Revenue Share by Segment (%)')
    axes[0].set_ylabel('Revenue Share (%)')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=0)
    for bar, val in zip(axes[0].patches, pct.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val}%', ha='center', fontsize=10)

    # Customer count by segment
    cnt = df_clusters['segment'].value_counts()
    colors2 = ['gold' if s == 'INSIDERS' else 'steelblue' for s in cnt.index]
    cnt.plot(kind='bar', ax=axes[1], color=colors2, edgecolor='white')
    axes[1].set_title('Customer Count by Segment')
    axes[1].set_ylabel('Customers')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=0)

    plt.suptitle('Business Results — Segment Analysis', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES / '04_segment_business_results.png', dpi=120)
    plt.show()

In [ ]:
if clusters_path.exists() and 'segment' in df_clusters.columns and agg_cols:
    seg_profile = df_clusters.groupby('segment')[agg_cols].median().round(1)
    seg_profile['n_customers'] = df_clusters.groupby('segment').size()
    if 'monetary' in df_clusters.columns:
        total = df_clusters['monetary'].sum()
        seg_profile['revenue_share_%'] = (
            df_clusters.groupby('segment')['monetary'].sum() / total * 100
        ).round(1)
    print('Full Segment Profile (median values):')
    display(seg_profile.sort_values('revenue_share_%', ascending=False)
            if 'revenue_share_%' in seg_profile.columns else seg_profile)

## 7. Business Recommendations

| Segment | Action | Goal |
|---|---|---|
| **INSIDERS** | Exclusive benefits, early access, premium support | Retain and upsell |
| **Loyalists** | Nurture programmes, loyalty points | Move to Insiders |
| **At-Risk** | Reactivation emails, discount vouchers | Prevent churn |
| **Churned** | Win-back campaign or suppress from budget | Minimal investment |

---
**Next notebook:** `05_deployment_and_consumption.ipynb` — serve the model via API and export for BI.